# 📊 Large Search Multi-Key — 1,500 req/min Demo

This notebook demonstrates a **production-grade multi-API-key pattern** for searching financial news at high throughput from a **single IP address**.

The Bigdata.com API enforces **500 requests/minute per API key**. By routing requests across **3 keys** with independent rate limiters and round-robin selection, this demo achieves **~1,425 requests/minute aggregate** (>1,400, safely under 1,500).

## Features
- **Multi-Key Pool**: Round-robin routing across 3 API keys with equal consumption
- **Per-Key Rate Limiting**: 475 req/min per key (5% safety margin from 500/key limit)
- **Entity Resolution**: Lookup tickers via Knowledge Graph API with CSV caching
- **Parallel Processing**: ThreadPoolExecutor with 36 search workers
- **SQLite Storage**: Persistent storage for search results
- **Query Interface**: Filter by ticker or topic

## Workflow
1. Configure inputs (tickers, dates, topics) and load 3 API keys from `.env`
2. Resolve tickers to entity IDs (with caching)
3. Execute parallel topic searches via the multi-key pool
4. Store results in SQLite database
5. Query results and review throughput metrics

Compare with [`large_search.ipynb`](large_search.ipynb) for the single-key (~460 req/min) baseline.

---


## 1️⃣ Configuration & User Inputs

**Modify the values below to customize your search:**


In [1]:
# =============================================================================
# 📝 USER INPUTS - MODIFY THESE VALUES
# =============================================================================

# Smaller Set: for test 
#TICKERS_INPUT = "AAPL, MSFT, GOOGL, TSLA, NVDA"

TICKERS_INPUT = "AAPL,ABNB,ADBE,ADI,ADP,ADSK,AEP,ALNY,AMAT,AMD,AMGN,AMZN,ANSS,APP,ARM,ASML,AVGO,AXON,AZN,BKR,BKNG,CDNS,CEG,CHTR,CMCSA,COST,CPRT,CRWD,CSCO,CSGP,CSX,CTAS,CTSH,DASH,DDOG,DXCM,EA,EXC,FANG,FAST,FER,FTNT,GEHC,GILD,GOOGL,HON,IDXX,INTC,INTU,ISRG,KDP,KHC,KLAC,LRCX,LIN,MAR,MCHP,MDLZ,MELI,META,MNST,MPWR,MSFT,MSTR,MU,NFLX,NVDA,NXPI,ODFL,ORLY,PANW,PAYX,PCAR,PDD,PEP,PLTR,PYPL,QCOM,REGN,ROP,ROST,SBUX,SHOP,SNPS,STX,TEAM,TMUS,TSLA,TTWO,TXN,VRSK,VRTX,VSNT,WBD,WDAY,WDC,XEL,ZS"


# Date range for search (YYYY-MM-DD format)
START_DATE = "2025-01-01"
END_DATE = "2025-01-12"

# Search topics - customize these for your research needs
# Use {company} placeholder for company name substitution
TOPICS = [
    {
      "topic_name": "Financial Metrics",
      "topic_text": "{company} earnings results revenue profit margin EPS"
    },
    {
      "topic_name": "Financial Metrics",
      "topic_text": "{company} financial performance growth decline quarterly annual"
    },
    {
      "topic_name": "Financial Metrics",
      "topic_text": "{company} guidance outlook forecast revision raised lowered"
    },
    {
      "topic_name": "M&A",
      "topic_text": "{company} strategy pivot restructuring transformation initiative"
    },
    {
      "topic_name": "M&A",
      "topic_text": "{company} acquisition merger divestiture deal buyout takeover"
    },
    {
      "topic_name": "Leadership",
      "topic_text": "{company} CEO CFO executive appointment resignation departure"
    },
    {
      "topic_name": "Competition",
      "topic_text": "{company} contract win loss award renewal partnership"
    },
    {
      "topic_name": "Competition",
      "topic_text": "{company} market share gain loss competitive position ranking"
    },
    {
      "topic_name": "Competition",
      "topic_text": "{company} competitor threat response pricing pressure disruption"
    },
    {
      "topic_name": "Products",
      "topic_text": "{company} product launch new release pipeline innovation R&D"
    },
    {
      "topic_name": "Supply Chain",
      "topic_text": "{company} operational disruption outage capacity shutdown closure"
    },
    {
      "topic_name": "Supply Chain",
      "topic_text": "{company} supply chain shortage constraint logistics inventory"
    },
    {
      "topic_name": "Supply Chain",
      "topic_text": "{company} production milestone output efficiency expansion ramp"
    },
    {
      "topic_name": "Costs",
      "topic_text": "{company} cost cutting layoffs restructuring expense reduction headcount"
    },
    {
      "topic_name": "Regulatory",
      "topic_text": "{company} regulatory approval FDA SEC FTC investigation ruling"
    },
    {
      "topic_name": "Regulatory",
      "topic_text": "{company} litigation lawsuit settlement verdict class action"
    },
    {
      "topic_name": "Industry",
      "topic_text": "{company} macroeconomic impact inflation recession tariff rates"
    },
    {
      "topic_name": "Industry",
      "topic_text": "{company} industry trend disruption shift demand outlook"
    },
    {
      "topic_name": "Financing",
      "topic_text": "{company} capital allocation capex investment spending expansion"
    },
    {
      "topic_name": "Financing",
      "topic_text": "{company} dividend buyback repurchase shareholder return payout"
    },
    {
      "topic_name": "Financing",
      "topic_text": "{company} debt issuance refinancing bond loan covenant"
    },
    {
      "topic_name": "Financing",
      "topic_text": "{company} credit rating downgrade upgrade outlook Moody's S&P"
    },
    {
      "topic_name": "Markets",
      "topic_text": "{company} investor sentiment analyst upgrade downgrade target"
    },
    {
      "topic_name": "Markets",
      "topic_text": "{company} catalyst event risk earnings conference investor day"
    },
    {
      "topic_name": "Markets",
      "topic_text": "{company} unusual trading volume insider transaction disclosure SEC filing"
    },
    {
      "topic_name": "Markets",
      "topic_text": "{company} activist investor hedge fund stake campaign proxy"
    }
  ]

# =============================================================================
# 🔧 ADVANCED SETTINGS (modify if needed)
# =============================================================================

# Per-key rate limiting (500 req/min per API key)
NUM_API_KEYS = 3
MAX_REQUESTS_PER_KEY_PER_MINUTE = 475  # 5% safety margin from 500/key
MAX_REQUESTS_PER_MINUTE = MAX_REQUESTS_PER_KEY_PER_MINUTE * NUM_API_KEYS  # 1,425 aggregate


# Max chunks per search query (higher = more results but slower)
MAX_CHUNKS_PER_QUERY = 10

# Document types to include
DOCUMENT_TYPES = ["NEWS"]

# Sentiment filter (positive/negative only for more relevant results)
SENTIMENT_VALUES = ["positive", "negative"]

# Parallel workers for entity resolution
ENTITY_WORKERS = 30

# Parallel workers for search - limits simultaneous connections
SEARCH_WORKERS = 36  # ~12 concurrent connections per key

# Max concurrent connections per API key
PER_KEY_MAX_CONCURRENT = SEARCH_WORKERS // NUM_API_KEYS  # 12 per key

# Sliding window parameters for burst prevention
WINDOW_SIZE_SECONDS = 5
MAX_RETRIES = 5  # sufficient with multi-layered protection

print("✅ Configuration loaded")


✅ Configuration loaded


## 2️⃣ Setup & Dependencies


In [2]:
import os
import time
import json
import sqlite3
import threading
import csv
from datetime import datetime, timezone, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, asdict
from typing import List, Dict, Optional, Tuple
from pathlib import Path
from collections import defaultdict

import requests
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# =============================================================================
# VALIDATE API KEYS (3 required for multi-key demo)
# =============================================================================
API_KEY_ENV_NAMES = ["BIGDATA_API_KEY1", "BIGDATA_API_KEY2", "BIGDATA_API_KEY3"]
API_KEYS = [os.getenv(name) for name in API_KEY_ENV_NAMES]

missing = [name for name, key in zip(API_KEY_ENV_NAMES, API_KEYS) if not key]
if missing:
    raise ValueError(
        f"❌ Missing API keys: {', '.join(missing)}\n"
        "Please create a .env file with:\n"
        "BIGDATA_API_KEY1=your_api_key_1\n"
        "BIGDATA_API_KEY2=your_api_key_2\n"
        "BIGDATA_API_KEY3=your_api_key_3"
    )


def mask_api_key(key: str) -> str:
    """Return masked API key for safe logging."""
    if len(key) <= 12:
        return "***"
    return f"{key[:8]}...{key[-4:]}"


# =============================================================================
# CONSTANTS & PATHS
# =============================================================================
BASE_URL = "https://api.bigdata.com/v1"
OUTPUT_DIR_NAME = "output_multi_key"
OUTPUT_DIR = Path(OUTPUT_DIR_NAME)
OUTPUT_DIR.mkdir(exist_ok=True)

# Share entity cache with single-key notebook (entity IDs are key-agnostic)
ENTITY_CACHE_FILE = Path("output") / "entity_cache.csv"
ENTITY_CACHE_FILE.parent.mkdir(exist_ok=True)

# SQLite database for search results
DATABASE_FILE = OUTPUT_DIR / "search_results.db"

# Parse tickers from input string
if TICKERS_INPUT:
    TICKERS = [t.strip().upper() for t in TICKERS_INPUT.split(",") if t.strip()]
    USE_ENTITY_IDS_DIRECTLY = False
else:
    TICKERS = []
    USE_ENTITY_IDS_DIRECTLY = True

# Pipeline timing (set before entity resolution, read in performance summary)
pipeline_start: float | None = None
entity_resolution_elapsed: float = 0.0
search_elapsed: float = 0.0

print("✅ Dependencies loaded successfully")
print("✅ API Keys configured:")
for env_name, key in zip(API_KEY_ENV_NAMES, API_KEYS):
    print(f"   • {env_name}: {mask_api_key(key)}")
print(f"\n📁 Output directory: {OUTPUT_DIR_NAME}")
if USE_ENTITY_IDS_DIRECTLY:
    print(f"📊 Using entity IDs from CSV")
else:
    print(f"📊 Tickers to search: {len(TICKERS)} tickers")
print(f"📅 Date range: {START_DATE} to {END_DATE}")
print(f"📝 Topics configured: {len(TOPICS)}")
print(f"🔑 Multi-key pool: {NUM_API_KEYS} keys × {MAX_REQUESTS_PER_KEY_PER_MINUTE} req/min = {MAX_REQUESTS_PER_MINUTE} req/min aggregate")


✅ Dependencies loaded successfully
✅ API Keys configured:
   • BIGDATA_API_KEY1: bd_v2_Fo...cu54
   • BIGDATA_API_KEY2: bd_v2_oV...yj45
   • BIGDATA_API_KEY3: bd_v2_iV...5e58

📁 Output directory: output_multi_key
📊 Tickers to search: 98 tickers
📅 Date range: 2025-01-01 to 2025-01-12
📝 Topics configured: 26
🔑 Multi-key pool: 3 keys × 475 req/min = 1425 req/min aggregate


## 3️⃣ Multi-Key Rate Limiting & Protection

Production-grade multi-key pattern with per-key isolation:

### 🔑 Layer 0: API Key Pool (Round-Robin)
- **3 API keys** each with dedicated session, rate limiter, and concurrency semaphore
- **Round-robin selection** ensures equal key consumption (~33.3% each)
- Retries stay on the same key for accurate per-key accounting

### 🛡️ Layer 1: Per-Key Sliding Window Rate Limiter
- **475 requests/min per key** (5% safety margin from 500/key API limit)
- **1,425 requests/min aggregate** across all keys from a single IP
- **5-second windows** to prevent request bursts that trigger WAF

### 🔒 Layer 2: Per-Key Concurrency Semaphore
- **12 simultaneous connections per key** (36 total)
- Prevents connection spikes that can trigger security filters

### 🔄 Layer 3: Automatic Retry Logic
- **Up to 5 retry attempts** with intelligent backoff
- **1-second wait** on 429 (rate limit) errors
- **Progressive backoff** on 403 (WAF) errors

**Key Insight:** Each API key has its own 500 req/min quota. Round-robin routing lets a single client process saturate ~1,425 req/min without exceeding any individual key limit.


In [3]:
from collections import deque


class SlidingWindowRateLimiter:
    """
    Thread-safe sliding window rate limiter with burst prevention.
    Uses 5-second windows to prevent request bursts.
    """

    def __init__(self, max_requests: int = 475, period_seconds: int = 60, window_size: int = 5):
        self.max_requests = max_requests
        self.period_seconds = period_seconds
        self.window_size = window_size
        self.max_per_window = int(max_requests * window_size / period_seconds)

        self.request_times = deque()
        self._lock = threading.Lock()

        self.total_requests = 0
        self.total_wait_time = 0.0
        self.throttle_events = 0
        self.rate_limit_warnings = 0

    def _clean_old_requests(self, current_time: float) -> None:
        cutoff_time = current_time - self.period_seconds
        while self.request_times and self.request_times[0] < cutoff_time:
            self.request_times.popleft()

    def _requests_in_window(self, current_time: float) -> int:
        window_start = current_time - self.window_size
        return sum(1 for t in self.request_times if t >= window_start)

    def acquire(self, timeout: float = 60.0) -> float:
        start_time = time.time()
        total_wait = 0.0

        while True:
            with self._lock:
                current_time = time.time()
                self._clean_old_requests(current_time)

                requests_in_period = len(self.request_times)
                requests_in_window = self._requests_in_window(current_time)

                if requests_in_period < self.max_requests and requests_in_window < self.max_per_window:
                    self.request_times.append(current_time)
                    self.total_requests += 1
                    self.total_wait_time += total_wait
                    return total_wait

                self.throttle_events += 1
                if requests_in_window >= self.max_per_window:
                    wait_time = self.window_size / 10
                else:
                    oldest_request = self.request_times[0]
                    wait_time = (oldest_request + self.period_seconds - current_time) + 0.1

            if time.time() - start_time > timeout:
                raise TimeoutError("Rate limiter timeout exceeded")

            time.sleep(min(wait_time, 1.0))
            total_wait += wait_time

    def get_stats(self) -> dict:
        with self._lock:
            return {
                "total_requests": self.total_requests,
                "total_wait_time_seconds": round(self.total_wait_time, 2),
                "throttle_events": self.throttle_events,
                "rate_limit_warnings": self.rate_limit_warnings,
                "current_requests_in_period": len(self.request_times),
                "max_requests_per_period": self.max_requests,
                "max_requests_per_window": self.max_per_window,
            }


class ConcurrencySemaphore:
    """Semaphore to limit simultaneous connections per API key."""

    def __init__(self, max_concurrent: int = 12):
        self.semaphore = threading.Semaphore(max_concurrent)
        self.max_concurrent = max_concurrent
        self.active_count = 0
        self._lock = threading.Lock()

        self.total_acquisitions = 0
        self.peak_concurrent = 0

    def __enter__(self):
        self.semaphore.acquire()
        with self._lock:
            self.active_count += 1
            self.total_acquisitions += 1
            self.peak_concurrent = max(self.peak_concurrent, self.active_count)
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        with self._lock:
            self.active_count -= 1
        self.semaphore.release()
        return False

    def get_stats(self) -> dict:
        with self._lock:
            return {
                "active_connections": self.active_count,
                "max_concurrent": self.max_concurrent,
                "total_acquisitions": self.total_acquisitions,
                "peak_concurrent": self.peak_concurrent,
            }


@dataclass
class ApiKeyClient:
    """Single API key with dedicated session, rate limiter, and concurrency control."""

    key_id: str
    env_name: str
    api_key: str
    session: requests.Session
    rate_limiter: SlidingWindowRateLimiter
    concurrency: ConcurrencySemaphore


class ApiKeyPool:
    """
    Round-robin pool of API key clients for equal key consumption.
    Each key has its own rate limiter (475 req/min) and concurrency semaphore.
    """

    def __init__(
        self,
        api_keys: list[str],
        env_names: list[str],
        max_requests_per_key: int,
        max_concurrent_per_key: int,
        window_size: int = 5,
    ):
        self.clients: list[ApiKeyClient] = []
        for i, (env_name, api_key) in enumerate(zip(env_names, api_keys)):
            self.clients.append(
                ApiKeyClient(
                    key_id=f"KEY{i + 1}",
                    env_name=env_name,
                    api_key=api_key,
                    session=requests.Session(),
                    rate_limiter=SlidingWindowRateLimiter(
                        max_requests=max_requests_per_key,
                        period_seconds=60,
                        window_size=window_size,
                    ),
                    concurrency=ConcurrencySemaphore(max_concurrent=max_concurrent_per_key),
                )
            )
        self._index = 0
        self._lock = threading.Lock()

    def select_client(self) -> ApiKeyClient:
        """Round-robin client selection (thread-safe)."""
        with self._lock:
            client = self.clients[self._index]
            self._index = (self._index + 1) % len(self.clients)
        return client

    def get_aggregate_stats(self) -> dict:
        """Per-key and aggregate statistics."""
        per_key = []
        total_requests = 0
        total_throttle = 0
        total_wait = 0.0
        total_warnings = 0
        peak_concurrent = 0

        for client in self.clients:
            rate_stats = client.rate_limiter.get_stats()
            conc_stats = client.concurrency.get_stats()
            reqs = rate_stats["total_requests"]
            total_requests += reqs
            total_throttle += rate_stats["throttle_events"]
            total_wait += rate_stats["total_wait_time_seconds"]
            total_warnings += rate_stats["rate_limit_warnings"]
            peak_concurrent = max(peak_concurrent, conc_stats["peak_concurrent"])
            per_key.append(
                {
                    "key_id": client.key_id,
                    "env_name": client.env_name,
                    "masked_key": mask_api_key(client.api_key),
                    "requests": reqs,
                    "throttle_events": rate_stats["throttle_events"],
                    "peak_concurrent": conc_stats["peak_concurrent"],
                }
            )

        for entry in per_key:
            entry["share_pct"] = round(100.0 * entry["requests"] / total_requests, 1) if total_requests else 0.0

        return {
            "total_requests": total_requests,
            "total_throttle_events": total_throttle,
            "total_wait_time_seconds": round(total_wait, 2),
            "total_rate_limit_warnings": total_warnings,
            "peak_concurrent": peak_concurrent,
            "max_concurrent_per_key": self.clients[0].concurrency.max_concurrent if self.clients else 0,
            "per_key": per_key,
        }

    def close(self) -> None:
        """Close all HTTP sessions."""
        for client in self.clients:
            client.session.close()


# Initialize multi-key pool
api_key_pool = ApiKeyPool(
    api_keys=API_KEYS,
    env_names=API_KEY_ENV_NAMES,
    max_requests_per_key=MAX_REQUESTS_PER_KEY_PER_MINUTE,
    max_concurrent_per_key=PER_KEY_MAX_CONCURRENT,
    window_size=WINDOW_SIZE_SECONDS,
)

print(f"✅ Multi-key API pool initialized:")
print(f"   • {NUM_API_KEYS} API keys with round-robin routing")
print(f"   • {MAX_REQUESTS_PER_KEY_PER_MINUTE} req/min per key ({MAX_REQUESTS_PER_MINUTE} aggregate)")
print(f"   • {WINDOW_SIZE_SECONDS}s sliding windows (~{api_key_pool.clients[0].rate_limiter.max_per_window} req/window per key)")
print(f"   • {PER_KEY_MAX_CONCURRENT} concurrent connections per key ({SEARCH_WORKERS} total)")


✅ Multi-key API pool initialized:
   • 3 API keys with round-robin routing
   • 475 req/min per key (1425 aggregate)
   • 5s sliding windows (~39 req/window per key)
   • 12 concurrent connections per key (36 total)


## 4️⃣ Entity Cache (CSV-based)

Persistent cache for entity IDs to avoid redundant API calls. The cache is stored in a CSV file and persists between notebook runs.

For production use, it can be in Redis or DB of choice. 


In [4]:
@dataclass
class EntityData:
    """Entity data structure for caching."""
    ticker: str
    entity_id: str
    company_name: str
    cached_at: str


class EntityCache:
    """
    CSV-based entity cache for persistent storage.
    Thread-safe for concurrent access.
    """
    
    def __init__(self, cache_file: Path):
        self.cache_file = cache_file
        self._cache: Dict[str, EntityData] = {}
        self._lock = threading.Lock()
        self._load_cache()
    
    def _load_cache(self) -> None:
        """Load existing cache from CSV file."""
        if not self.cache_file.exists():
            print(f"📝 Creating new entity cache: {self.cache_file}")
            return
        
        try:
            with open(self.cache_file, 'r', newline='', encoding='utf-8') as f:
                reader = csv.DictReader(f)
                for row in reader:
                    ticker = row['ticker'].upper()
                    self._cache[ticker] = EntityData(
                        ticker=ticker,
                        entity_id=row['entity_id'],
                        company_name=row['company_name'],
                        cached_at=row['cached_at']
                    )
            print(f"📂 Loaded {len(self._cache)} entities from cache")
        except Exception as e:
            print(f"⚠️ Error loading cache: {e}")
    
    def _save_cache(self) -> None:
        """Save cache to CSV file."""
        try:
            with open(self.cache_file, 'w', newline='', encoding='utf-8') as f:
                fieldnames = ['ticker', 'entity_id', 'company_name', 'cached_at']
                writer = csv.DictWriter(f, fieldnames=fieldnames)
                writer.writeheader()
                for entity in self._cache.values():
                    writer.writerow(asdict(entity))
        except Exception as e:
            print(f"⚠️ Error saving cache: {e}")
    
    def get(self, ticker: str) -> Optional[EntityData]:
        """Get entity from cache."""
        with self._lock:
            return self._cache.get(ticker.upper())
    
    def set(self, ticker: str, entity_id: str, company_name: str) -> EntityData:
        """Add entity to cache and persist to file."""
        with self._lock:
            entity = EntityData(
                ticker=ticker.upper(),
                entity_id=entity_id,
                company_name=company_name,
                cached_at=datetime.now().isoformat()
            )
            self._cache[ticker.upper()] = entity
            self._save_cache()
            return entity
    
    def get_all(self) -> Dict[str, EntityData]:
        """Get all cached entities."""
        with self._lock:
            return dict(self._cache)
    
    def __len__(self) -> int:
        return len(self._cache)


# Initialize entity cache
entity_cache = EntityCache(ENTITY_CACHE_FILE)
print(f"✅ Entity cache initialized with {len(entity_cache)} entries")


📝 Creating new entity cache: output/entity_cache.csv
✅ Entity cache initialized with 0 entries


## 5️⃣ API Client Functions

Core functions for making API requests with rate limiting and retry logic.


In [5]:
def make_api_request(
    endpoint: str,
    payload: dict,
    max_retries: int = MAX_RETRIES,
    base_delay: float = 1.0,
    client: ApiKeyClient | None = None,
) -> Optional[dict]:
    """
    Make an API request via the multi-key pool with per-key rate limiting.

    Selects a client via round-robin on first attempt; retries stay on the same key.
    """
    url = f"{BASE_URL}{endpoint}"
    if client is None:
        client = api_key_pool.select_client()

    retry_count = 0
    warning_interval = 100

    while retry_count < max_retries:
        with client.concurrency:
            try:
                client.rate_limiter.acquire(timeout=120.0)

                response = client.session.post(
                    url,
                    headers={
                        "X-API-KEY": client.api_key,
                        "Content-Type": "application/json",
                    },
                    json=payload,
                    timeout=120,
                )

                if response.status_code == 200:
                    return response.json()

                elif response.status_code == 429:
                    retry_count += 1
                    if retry_count % warning_interval == 0:
                        client.rate_limiter.rate_limit_warnings += 1
                        print(f"⚠️ [{client.key_id}] Rate limit warning: {retry_count} retries (429)")
                    time.sleep(1.0)
                    continue

                elif response.status_code == 403:
                    retry_count += 1
                    if retry_count % warning_interval == 0:
                        client.rate_limiter.rate_limit_warnings += 1
                        print(f"🚨 [{client.key_id}] Security filter warning: {retry_count} retries (403)")
                    delay = min(base_delay * (1.5 ** (retry_count // 10)), 5.0)
                    time.sleep(delay)
                    continue

                else:
                    retry_count += 1
                    if retry_count < max_retries:
                        delay = min(base_delay * (2 ** min(retry_count, 10)), 30.0)
                        if retry_count % warning_interval == 0:
                            print(f"⚠️ [{client.key_id}] API error {response.status_code} after {retry_count} retries")
                        time.sleep(delay)
                        continue
                    print(f"❌ [{client.key_id}] Final API error {response.status_code}: {response.text[:200]}")
                    return None

            except requests.exceptions.Timeout:
                retry_count += 1
                if retry_count % warning_interval == 0:
                    print(f"⏱️ [{client.key_id}] Timeout after {retry_count} retries")
                if retry_count < max_retries:
                    time.sleep(base_delay * min(retry_count, 10))
                    continue
                return None

            except TimeoutError as e:
                print(f"⏰ [{client.key_id}] Rate limiter timeout: {e}")
                retry_count += 1
                time.sleep(5.0)
                continue

            except Exception as e:
                retry_count += 1
                if retry_count % warning_interval == 0:
                    print(f"❌ [{client.key_id}] Request error after {retry_count} retries: {e}")
                if retry_count < max_retries:
                    time.sleep(base_delay * min(retry_count, 10))
                    continue
                return None

    print(f"❌ [{client.key_id}] Exhausted all {max_retries} retry attempts")
    return None


def lookup_entity(ticker: str) -> Optional[EntityData]:
    """Look up entity ID for a ticker using the Knowledge Graph API."""
    ticker = ticker.upper()

    cached = entity_cache.get(ticker)
    if cached:
        return cached

    payload = {"query": ticker, "types": ["PUBLIC"]}
    response = make_api_request("/knowledge-graph/companies", payload, max_retries=100)

    if response and response.get("results"):
        company = response["results"][0]
        entity = entity_cache.set(
            ticker=ticker,
            entity_id=company["id"],
            company_name=company["name"],
        )
        print(f"✅ Resolved: {ticker} → {company['name']} ({company['id']})")
        return entity

    print(f"❌ Could not resolve ticker: {ticker}")
    return None


print("✅ API client functions defined with multi-key pool and retry logic")


✅ API client functions defined with multi-key pool and retry logic


## 6️⃣ Parallel Entity Resolution

Resolve all tickers to entity IDs in parallel for large portfolios.


In [ ]:
def resolve_entities_parallel(tickers: List[str], max_workers: int = 10) -> Dict[str, EntityData]:
    """
    Resolve multiple tickers to entity IDs in parallel using ThreadPoolExecutor.
    
    Args:
        tickers: List of ticker symbols
        max_workers: Number of parallel workers
        
    Returns:
        Dictionary of ticker -> EntityData
    """
    print(f"\n🔍 Resolving {len(tickers)} tickers...")
    start_time = time.time()
    
    results = {}
    failed = []
    
    # Check which tickers are already cached
    tickers_to_lookup = []
    for ticker in tickers:
        cached = entity_cache.get(ticker)
        if cached:
            results[ticker] = cached
            print(f"  📦 Cache hit: {ticker} → {cached.company_name}")
        else:
            tickers_to_lookup.append(ticker)
    
    if not tickers_to_lookup:
        print(f"\n✅ All {len(results)} tickers resolved from cache")
        return results
    
    print(f"\n🌐 Looking up {len(tickers_to_lookup)} tickers via API...")
    
    # Parallel lookup for tickers not in cache
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_ticker = {
            executor.submit(lookup_entity, ticker): ticker
            for ticker in tickers_to_lookup
        }
        
        for future in as_completed(future_to_ticker):
            ticker = future_to_ticker[future]
            try:
                entity = future.result()
                if entity:
                    results[ticker] = entity
                else:
                    failed.append(ticker)
            except Exception as e:
                print(f"❌ Error resolving {ticker}: {e}")
                failed.append(ticker)
    
    elapsed = time.time() - start_time
    print(f"\n📊 Entity Resolution Summary:")
    print(f"   ✅ Resolved: {len(results)} tickers")
    print(f"   ❌ Failed: {len(failed)} tickers {failed if failed else ''}")
    print(f"   ⏱️ Time: {elapsed:.2f}s")
    
    return results


# Execute entity resolution
pipeline_start = time.time()
entity_resolution_start = time.time()
entities = resolve_entities_parallel(TICKERS, max_workers=ENTITY_WORKERS)
entity_resolution_elapsed = time.time() - entity_resolution_start

# Display resolved entities
print("\n📋 Resolved Entities:")
for ticker, entity in entities.items():
    print(f"   {ticker}: {entity.company_name} ({entity.entity_id})")


## 7️⃣ SQLite Database Setup

Create database tables for storing search results.


In [7]:
def setup_database(db_path: Path) -> sqlite3.Connection:
    """
    Create SQLite database and tables for storing search results.
    
    Args:
        db_path: Path to database file
        
    Returns:
        Database connection
    """
    conn = sqlite3.connect(str(db_path), check_same_thread=False)
    cursor = conn.cursor()
    
    # Create main search_results table
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS search_results (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            search_id TEXT NOT NULL,
            ticker TEXT NOT NULL,
            company_name TEXT NOT NULL,
            entity_id TEXT NOT NULL,
            topic_name TEXT NOT NULL,
            topic_text TEXT NOT NULL,
            document_id TEXT NOT NULL,
            headline TEXT,
            timestamp TEXT,
            source_name TEXT,
            source_rank TEXT,
            document_url TEXT,
            document_type TEXT,
            chunk_text TEXT,
            chunk_relevance REAL,
            chunk_sentiment REAL,
            search_date TEXT NOT NULL,
            created_at TEXT DEFAULT CURRENT_TIMESTAMP,
            UNIQUE(search_id, document_id, topic_name)
        )
    """)
    
    # Create indexes for faster queries
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_ticker ON search_results(ticker)")
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_topic ON search_results(topic_name)")
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_search_id ON search_results(search_id)")
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_timestamp ON search_results(timestamp)")
    
    # Create search_runs table to track search history
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS search_runs (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            search_id TEXT UNIQUE NOT NULL,
            tickers TEXT NOT NULL,
            start_date TEXT NOT NULL,
            end_date TEXT NOT NULL,
            topics_count INTEGER,
            results_count INTEGER,
            duration_seconds REAL,
            created_at TEXT DEFAULT CURRENT_TIMESTAMP
        )
    """)
    
    conn.commit()
    return conn


# Initialize database
db_conn = setup_database(DATABASE_FILE)
print(f"✅ Database initialized: {DATABASE_FILE}")


✅ Database initialized: output_multi_key/search_results.db


## 8️⃣ Search Functions

Core search functionality with parallel execution using ThreadPoolExecutor.


In [8]:
def format_timestamp(dt: datetime) -> str:
    """Format datetime to ISO 8601 with milliseconds and Z suffix."""
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    iso_str = dt.strftime('%Y-%m-%dT%H:%M:%S')
    milliseconds = dt.microsecond // 1000
    return f"{iso_str}.{milliseconds:03d}Z"


def search_topic(
    entity: EntityData,
    topic: dict,
    start_date: str,
    end_date: str
) -> List[dict]:
    """
    Search for a single ticker + topic combination.
    
    Args:
        entity: EntityData for the ticker
        topic: Topic dict with topic_name and topic_text
        start_date: Start date (YYYY-MM-DD)
        end_date: End date (YYYY-MM-DD)
        
    Returns:
        List of search result dictionaries
    """
    # Format topic text with company name
    topic_text = topic["topic_text"].format(company=entity.company_name)
    
    # Parse dates and convert to UTC
    start_dt = datetime.strptime(start_date, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    end_dt = datetime.strptime(end_date, "%Y-%m-%d").replace(
        hour=23, minute=59, second=59, tzinfo=timezone.utc
    )
    
    # Build search payload
    payload = {
        "query": {
            "text": topic_text,
            "filters": {
                "timestamp": {
                    "start": format_timestamp(start_dt),
                    "end": format_timestamp(end_dt)
                },
                "entity": {
                    "all_of": [entity.entity_id]
                },
                "document_type": {
                    "mode": "INCLUDE",
                    "values": DOCUMENT_TYPES
                },
                "sentiment": {
                    "values": SENTIMENT_VALUES
                }
            },
            "max_chunks": MAX_CHUNKS_PER_QUERY
        }
    }
    
    response = make_api_request("/search", payload, max_retries=5)
    
    results = []
    if response and response.get("results"):
        for doc in response["results"]:
            # Get best chunk (highest relevance)
            chunks = doc.get("chunks", [])
            if not chunks:
                continue
            
            best_chunk = max(chunks, key=lambda c: c.get("relevance", 0))
            
            results.append({
                "ticker": entity.ticker,
                "company_name": entity.company_name,
                "entity_id": entity.entity_id,
                "topic_name": topic["topic_name"],
                "topic_text": topic_text,
                "document_id": doc.get("id", ""),
                "headline": doc.get("headline", ""),
                "timestamp": doc.get("timestamp", ""),
                "source_name": doc.get("source", {}).get("name", ""),
                "source_rank": doc.get("source", {}).get("rank", ""),
                "document_url": doc.get("url", ""),
                "document_type": doc.get("document_type", "NEWS"),
                "chunk_text": best_chunk.get("text", ""),
                "chunk_relevance": best_chunk.get("relevance", 0),
                "chunk_sentiment": best_chunk.get("sentiment", 0),
            })
    
    return results


print("✅ Search functions defined")


✅ Search functions defined


In [9]:
def execute_parallel_search(
    entities: Dict[str, EntityData],
    topics: List[dict],
    start_date: str,
    end_date: str,
    max_workers: int = 40
) -> Tuple[str, List[dict], float]:
    """
    Execute search for all ticker+topic combinations in parallel with robust rate limiting.
    
    Features:
    - Sliding window rate limiting (460 req/min with 5s windows)
    - Concurrency control (max 40 simultaneous connections)
    - Automatic retry with up to 10,000 attempts
    
    Args:
        entities: Dictionary of ticker -> EntityData
        topics: List of topic dictionaries
        start_date: Start date (YYYY-MM-DD)
        end_date: End date (YYYY-MM-DD)
        max_workers: Number of parallel workers (also controls max simultaneous connections)
        
    Returns:
        Tuple of (search_id, list of all results, duration)
    """
    search_id = f"search_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    
    # Build all search task combinations
    search_tasks = []
    for ticker, entity in entities.items():
        for topic in topics:
            search_tasks.append((entity, topic))
    
    total_tasks = len(search_tasks)
    print(f"\n🚀 Starting parallel search with multi-layered protection")
    print(f"   📊 Search ID: {search_id}")
    print(f"   📈 Tickers: {len(entities)}")
    print(f"   📝 Topics: {len(topics)}")
    print(f"   🔢 Total queries: {total_tasks}")
    print(f"   👷 Workers: {max_workers}")
    print(f"   ⏱️ Rate limit: {MAX_REQUESTS_PER_MINUTE} req/min aggregate ({MAX_REQUESTS_PER_KEY_PER_MINUTE}/key)")
    print(f"   🔒 Concurrency limit: {SEARCH_WORKERS} simultaneous connections")
    print(f"   🔄 Auto-retry: Up to {MAX_RETRIES} attempts per request")
    print()
    
    start_time = time.time()
    all_results = []
    completed = 0
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_task = {
            executor.submit(
                search_topic, entity, topic, start_date, end_date
            ): (entity, topic)
            for entity, topic in search_tasks
        }
        
        for future in as_completed(future_to_task):
            entity, topic = future_to_task[future]
            completed += 1
            
            try:
                results = future.result()
                if results:
                    all_results.extend(results)
                    print(
                        f"   [{completed}/{total_tasks}] {entity.ticker} + {topic['topic_name']}: "
                        f"{len(results)} results"
                    )
                else:
                    print(
                        f"   [{completed}/{total_tasks}] {entity.ticker} + {topic['topic_name']}: "
                        f"0 results"
                    )
            except Exception as e:
                print(f"   ❌ [{completed}/{total_tasks}] {entity.ticker} + {topic['topic_name']}: Error - {e}")
    
    elapsed = time.time() - start_time
    
    # Get comprehensive stats from all three protection layers
    pool_stats = api_key_pool.get_aggregate_stats()
    
    print(f"\n📊 Search Complete!")
    print(f"   ✅ Total results: {len(all_results)}")
    print(f"   ⏱️ Duration: {elapsed:.2f}s")
    print(f"   📈 Rate: {total_tasks / elapsed:.1f} queries/sec")
    
    print(f"\n🔧 Multi-Key Pool Stats:")
    print(f"   📊 Total requests: {pool_stats['total_requests']}")
    print(f"   📈 Effective rate: {pool_stats['total_requests'] / elapsed * 60:.0f} req/min")
    for pk in pool_stats["per_key"]:
        print(f"      • {pk['key_id']} ({pk['masked_key']}): {pk['requests']} requests ({pk['share_pct']}%)")
    print(f"   ⏸️ Throttle events: {pool_stats['total_throttle_events']}")
    print(f"   ⚠️ Rate limit warnings: {pool_stats['total_rate_limit_warnings']}")
    print(f"   🔒 Peak concurrent: {pool_stats['peak_concurrent']}/{SEARCH_WORKERS}")
    
    return search_id, all_results, elapsed


print("✅ Parallel search executor defined with robust multi-layered protection")


✅ Parallel search executor defined with robust multi-layered protection


## 9️⃣ Execute Search & Store Results

Run the search and save results to SQLite database.


In [10]:
def save_results_to_db(
    conn: sqlite3.Connection,
    search_id: str,
    results: List[dict],
    tickers: List[str],
    start_date: str,
    end_date: str,
    topics_count: int,
    duration: float
) -> int:
    """
    Save search results to SQLite database.
    
    Args:
        conn: Database connection
        search_id: Unique search identifier
        results: List of result dictionaries
        tickers: List of tickers searched
        start_date: Search start date
        end_date: Search end date
        topics_count: Number of topics
        duration: Search duration in seconds
        
    Returns:
        Number of rows inserted
    """
    cursor = conn.cursor()
    
    # Insert search run metadata
    cursor.execute("""
        INSERT OR REPLACE INTO search_runs 
        (search_id, tickers, start_date, end_date, topics_count, results_count, duration_seconds)
        VALUES (?, ?, ?, ?, ?, ?, ?)
    """, (search_id, ",".join(tickers), start_date, end_date, topics_count, len(results), duration))
    
    # Insert results
    inserted = 0
    search_date = datetime.now().isoformat()
    
    for result in results:
        try:
            cursor.execute("""
                INSERT OR IGNORE INTO search_results (
                    search_id, ticker, company_name, entity_id, topic_name, topic_text,
                    document_id, headline, timestamp, source_name, source_rank,
                    document_url, document_type, chunk_text, chunk_relevance,
                    chunk_sentiment, search_date
                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                search_id,
                result["ticker"],
                result["company_name"],
                result["entity_id"],
                result["topic_name"],
                result["topic_text"],
                result["document_id"],
                result["headline"],
                result["timestamp"],
                result["source_name"],
                result["source_rank"],
                result["document_url"],
                result["document_type"],
                result["chunk_text"],
                result["chunk_relevance"],
                result["chunk_sentiment"],
                search_date
            ))
            inserted += cursor.rowcount
        except Exception as e:
            print(f"⚠️ Error inserting result: {e}")
    
    conn.commit()
    return inserted


print("✅ Database save function defined")


✅ Database save function defined


### Execute the search

In [ ]:
# =============================================================================
# 🚀 EXECUTE THE SEARCH
# =============================================================================

if entities:
    # Execute parallel search
    search_id, search_results, duration = execute_parallel_search(
        entities=entities,
        topics=TOPICS,
        start_date=START_DATE,
        end_date=END_DATE,
        max_workers=SEARCH_WORKERS,
    )
    search_elapsed = duration
    
    # Save results to database
    if search_results:
        rows_inserted = save_results_to_db(
            conn=db_conn,
            search_id=search_id,
            results=search_results,
            tickers=list(entities.keys()),
            start_date=START_DATE,
            end_date=END_DATE,
            topics_count=len(TOPICS),
            duration=duration
        )
        print(f"\n💾 Saved {rows_inserted} results to database")
        #print(f"📁 Database file: {DATABASE_FILE.absolute()}")
    else:
        print("\n⚠️ No results to save")
else:
    print("❌ No entities resolved. Cannot execute search.")


---

## 🔟 Query Results by Ticker

Retrieve and display results for a specific ticker from the database.


In [12]:
def query_by_ticker(conn: sqlite3.Connection, ticker: str, limit: int = 50) -> List[dict]:
    """
    Query search results for a specific ticker.
    
    Args:
        conn: Database connection
        ticker: Stock ticker symbol
        limit: Maximum results to return
        
    Returns:
        List of result dictionaries
    """
    cursor = conn.cursor()
    
    cursor.execute("""
        SELECT 
            ticker, company_name, topic_name, headline, timestamp,
            source_name, chunk_relevance, chunk_sentiment, chunk_text, document_url
        FROM search_results
        WHERE ticker = ?
        ORDER BY timestamp DESC, chunk_relevance DESC
        LIMIT ?
    """, (ticker.upper(), limit))
    
    columns = [desc[0] for desc in cursor.description]
    results = [dict(zip(columns, row)) for row in cursor.fetchall()]
    
    return results


# =============================================================================
# 🔍 QUERY BY TICKER - Change the ticker below to query different stocks
# =============================================================================
QUERY_TICKER = "AAPL"  # <-- MODIFY THIS

ticker_results = query_by_ticker(db_conn, QUERY_TICKER, limit=20)

print(f"\n📊 Results for {QUERY_TICKER}: {len(ticker_results)} articles")
print("=" * 80)

for i, result in enumerate(ticker_results[:10], 1):
    sentiment_emoji = "🟢" if result['chunk_sentiment'] and result['chunk_sentiment'] > 0 else "🔴" if result['chunk_sentiment'] and result['chunk_sentiment'] < 0 else "⚪"
    print(f"\n{i}. [{result['topic_name']}] {sentiment_emoji}")
    headline = result.get('headline', 'No headline')
    print(f"   📰 {headline[:80]}{'...' if len(headline) > 80 else ''}")
    print(f"   📅 {result['timestamp']} | 📡 {result['source_name']}")
    sentiment_str = f"{result['chunk_sentiment']:.2f}" if result['chunk_sentiment'] else "N/A"
    relevance_str = f"{result['chunk_relevance']:.3f}" if result['chunk_relevance'] else "N/A"
    print(f"   📈 Relevance: {relevance_str} | Sentiment: {sentiment_str}")
    chunk = result.get('chunk_text', '')
    print(f"   📝 {chunk[:150]}...")



📊 Results for AAPL: 20 articles

1. [M&A] 🔴
   📰 Mark Zuckerberg takes another shot at Apple and says it's 'squeezing people' for...
   📅 2025-01-12T21:21:13 | 📡 Yahoo! Finance
   📈 Relevance: 0.242 | Sentiment: -0.25
   📝 The CEO said Apple declined to let Meta use the same "protocol" Apple uses for Airpods in order to allow the Meta glasses to connect seamlessly to iPh...

2. [Markets] 🔴
   📰 Apple wants to keep its diversity programmes from being disavowed
   📅 2025-01-12T17:40:48 | 📡 Business Standard
   📈 Relevance: 0.144 | Sentiment: -0.75
   📝 The National Center for Public Policy, a conservative think tank, submitted a proposal that the company consider abolishing its "Inclusion & Diversity...

3. [Markets] 🔴
   📰 Apple Joins Costco In Opposing Conservative Meddling In DEI Programs
   📅 2025-01-12T17:16:28 | 📡 Black Enterprise
   📈 Relevance: 0.157 | Sentiment: -0.67
   📝 On Jan. 10, Apple filed its annual company proxy report for shareholders in advance of its shareholder mee

## 1️⃣1️⃣ Query Results by Topic

Retrieve and display results for a specific topic across all tickers.


In [13]:
def query_by_topic(conn: sqlite3.Connection, topic_name: str, limit: int = 50) -> List[dict]:
    """
    Query search results for a specific topic across all tickers.
    
    Args:
        conn: Database connection
        topic_name: Topic name to search
        limit: Maximum results to return
        
    Returns:
        List of result dictionaries
    """
    cursor = conn.cursor()
    
    cursor.execute("""
        SELECT 
            ticker, company_name, topic_name, headline, timestamp,
            source_name, chunk_relevance, chunk_sentiment, chunk_text, document_url
        FROM search_results
        WHERE topic_name = ?
        ORDER BY timestamp DESC, chunk_relevance DESC
        LIMIT ?
    """, (topic_name, limit))
    
    columns = [desc[0] for desc in cursor.description]
    results = [dict(zip(columns, row)) for row in cursor.fetchall()]
    
    return results


def get_available_topics(conn: sqlite3.Connection) -> List[str]:
    """Get list of all topics in the database."""
    cursor = conn.cursor()
    cursor.execute("SELECT DISTINCT topic_name FROM search_results ORDER BY topic_name")
    return [row[0] for row in cursor.fetchall()]


# Show available topics
available_topics = get_available_topics(db_conn)
print("📋 Available topics in database:")
for topic in available_topics:
    print(f"   • {topic}")


📋 Available topics in database:
   • Competition
   • Costs
   • Financial Metrics
   • Financing
   • Industry
   • Leadership
   • M&A
   • Markets
   • Products
   • Regulatory
   • Supply Chain


In [ ]:
# =============================================================================
# 🔍 QUERY BY TOPIC - Change the topic below to query different topics
# =============================================================================
QUERY_TOPIC = "Leadership"  # <-- MODIFY THIS
# Initialize database

topic_results = query_by_topic(db_conn, QUERY_TOPIC, limit=30)

print(f"\n📊 Results for topic '{QUERY_TOPIC}': {len(topic_results)} articles")
print("=" * 80)

# Group results by ticker for better readability
grouped = defaultdict(list)
for result in topic_results:
    grouped[result['ticker']].append(result)

for ticker, results in grouped.items():
    print(f"\n🏢 {ticker} ({results[0]['company_name']})")
    print("-" * 60)
    
    for result in results[:5]:  # Show top 5 per ticker
        sentiment_emoji = "🟢" if result['chunk_sentiment'] and result['chunk_sentiment'] > 0 else "🔴" if result['chunk_sentiment'] and result['chunk_sentiment'] < 0 else "⚪"
        headline = result.get('headline', 'No headline')
        print(f"   {sentiment_emoji} {headline[:70]}{'...' if len(headline) > 70 else ''}")
        relevance_str = f"{result['chunk_relevance']:.3f}" if result['chunk_relevance'] else "N/A"
        print(f"      📅 {result['timestamp']} | 📈 Rel: {relevance_str}")


## 1️⃣2️⃣ Summary Statistics

View aggregated statistics from the database.


In [15]:
def get_database_stats(conn: sqlite3.Connection) -> dict:
    """Get summary statistics from the database."""
    cursor = conn.cursor()
    
    # Total results
    cursor.execute("SELECT COUNT(*) FROM search_results")
    total_results = cursor.fetchone()[0]
    
    # Results by ticker
    cursor.execute("""
        SELECT ticker, company_name, COUNT(*) as count 
        FROM search_results 
        GROUP BY ticker 
        ORDER BY count DESC
    """)
    by_ticker = cursor.fetchall()
    
    # Results by topic
    cursor.execute("""
        SELECT topic_name, COUNT(*) as count 
        FROM search_results 
        GROUP BY topic_name 
        ORDER BY count DESC
    """)
    by_topic = cursor.fetchall()
    
    # Average sentiment by ticker
    cursor.execute("""
        SELECT ticker, AVG(chunk_sentiment) as avg_sentiment
        FROM search_results
        GROUP BY ticker
    """)
    sentiment_by_ticker = cursor.fetchall()
    
    return {
        "total_results": total_results,
        "by_ticker": by_ticker,
        "by_topic": by_topic,
        "sentiment_by_ticker": sentiment_by_ticker,
    }


# Display statistics
stats = get_database_stats(db_conn)

print("\n" + "=" * 80)
print("📊 DATABASE STATISTICS")
print("=" * 80)

print(f"\n📈 Total Results: {stats['total_results']}")

print(f"\n📊 Results by Ticker:")
for ticker, company, count in stats['by_ticker']:
    company_short = company[:25] if company else "Unknown"
    print(f"   {ticker:6} ({company_short:25}): {count:4} articles")

print(f"\n📝 Results by Topic:")
for topic, count in stats['by_topic']:
    print(f"   {topic:15}: {count:4} articles")

print(f"\n🎭 Average Sentiment by Ticker:")
for ticker, sentiment in stats['sentiment_by_ticker']:
    if sentiment:
        emoji = "🟢" if sentiment > 0.1 else "🔴" if sentiment < -0.1 else "⚪"
        print(f"   {ticker:6}: {emoji} {sentiment:.3f}")
    else:
        print(f"   {ticker:6}: ⚪ N/A")



📊 DATABASE STATISTICS

📈 Total Results: 3147

📊 Results by Ticker:
   AAPL   (Apple Inc.               ):  102 articles
   TSLA   (Tesla Inc.               ):   90 articles
   META   (Meta Platforms Inc.      ):   81 articles
   NVDA   (NVIDIA Corp.             ):   81 articles
   MSFT   (Microsoft Corp.          ):   75 articles
   INTC   (Intel Corp.              ):   73 articles
   PLTR   (Palantir Technologies Inc):   70 articles
   REGN   (Regeneron Pharmaceuticals):   66 articles
   MU     (Micron Technology Inc.   ):   65 articles
   AMD    (Advanced Micro Devices In):   62 articles
   ASML   (ASML Holding N.V.        ):   61 articles
   CEG    (Constellation Energy Corp):   60 articles
   AMZN   (Amazon.com Inc.          ):   59 articles
   AVGO   (Broadcom Inc.            ):   53 articles
   SBUX   (Starbucks Corp.          ):   52 articles
   PANW   (Palo Alto Networks Inc.  ):   51 articles
   CRWD   (CrowdStrike Holdings Inc.):   48 articles
   MDLZ   (Mondelez Internation

## 1️⃣3️⃣ Performance Summary — Multi-Key Throughput

Aggregate throughput metrics for the full API pipeline (entity resolution + search).


In [16]:
def format_duration(seconds: float) -> str:
    """Format seconds as human-readable duration."""
    if seconds < 60:
        return f"{seconds:.1f}s"
    minutes = int(seconds // 60)
    secs = seconds % 60
    return f"{minutes}m {secs:.0f}s"


if pipeline_start is None:
    pipeline_start = time.time()

pipeline_elapsed = entity_resolution_elapsed + search_elapsed
pool_stats = api_key_pool.get_aggregate_stats()
total_requests = pool_stats["total_requests"]
effective_rpm = (total_requests / pipeline_elapsed * 60) if pipeline_elapsed > 0 else 0.0

print("\n" + "=" * 80)
print("PERFORMANCE SUMMARY — Multi-Key Throughput Demo")
print("=" * 80)
print(f"Total API requests:     {total_requests:,}")
print(f"Wall-clock time:        {format_duration(pipeline_elapsed)}  (entity: {format_duration(entity_resolution_elapsed)} + search: {format_duration(search_elapsed)})")
print(f"Effective throughput:   {effective_rpm:,.0f} req/min")
print(f"Target aggregate cap:   {MAX_REQUESTS_PER_MINUTE:,} req/min ({MAX_REQUESTS_PER_KEY_PER_MINUTE}/key × {NUM_API_KEYS} keys)")
print()
print("Per-key distribution:")
for pk in pool_stats["per_key"]:
    print(f"  {pk['key_id']} ({pk['masked_key']}):  {pk['requests']:,} requests  ({pk['share_pct']}%)")
print()
print(f"Rate limit warnings (429/403): {pool_stats['total_rate_limit_warnings']}")
print(f"Peak concurrent connections:   {pool_stats['peak_concurrent']}/{SEARCH_WORKERS}")
print(f"Throttle events (brief pauses): {pool_stats['total_throttle_events']:,}")
print("=" * 80)



PERFORMANCE SUMMARY — Multi-Key Throughput Demo
Total API requests:     2,646
Wall-clock time:        3m 6s  (entity: 3.6s + search: 3m 3s)
Effective throughput:   851 req/min
Target aggregate cap:   1,425 req/min (475/key × 3 keys)

Per-key distribution:
  KEY1 (bd_v2_Fo...cu54):  882 requests  (33.3%)
  KEY2 (bd_v2_oV...yj45):  882 requests  (33.3%)
  KEY3 (bd_v2_iV...5e58):  882 requests  (33.3%)

Rate limit warnings (429/403): 0
Peak concurrent connections:   12/36
Throttle events (brief pauses): 590


In [17]:
# Close database connection
db_conn.close()
print("✅ Database connection closed")
api_key_pool.close()
print("✅ All API key sessions closed")

# Final summary
print("\n" + "=" * 80)
print("🎉 Multi-Key Large SEARCH COMPLETE!")
print("=" * 80)
print(f"   • Entity cache: {ENTITY_CACHE_FILE}")
print(f"   • Database: {DATABASE_FILE}")

pool_stats = api_key_pool.get_aggregate_stats()

print("\n" + "=" * 80)
print("🔧 MULTI-KEY PROTECTION FINAL STATS")
print("=" * 80)

print("\n🔑 API Key Pool (Round-Robin)")
print(f"   • Total API requests made: {pool_stats['total_requests']:,}")
print(f"   • Aggregate rate achieved: {(pool_stats['total_requests'] / pipeline_elapsed * 60):,.0f} req/min")
for pk in pool_stats["per_key"]:
    print(f"   • {pk['key_id']}: {pk['requests']:,} requests ({pk['share_pct']}%) — peak {pk['peak_concurrent']}/{PER_KEY_MAX_CONCURRENT} concurrent")

print("\n🛡️ Per-Key Rate Limiters")
print(f"   • Limit per key: {MAX_REQUESTS_PER_KEY_PER_MINUTE} req/min")
print(f"   • Throttle events (all keys): {pool_stats['total_throttle_events']:,}")
print(f"   • Total wait time: {pool_stats['total_wait_time_seconds']:.2f}s")
print(f"   • Rate limit warnings (429/403): {pool_stats['total_rate_limit_warnings']:,}")

print("\n🔄 Automatic Retry")
print(f"   • Max retry attempts configured: {MAX_RETRIES}")
print(f"   • Retry strategy: 1s wait on 429, progressive backoff on 403")

print("\n✅ All protection layers worked successfully — multi-key pattern ready for production use!")


✅ Database connection closed
✅ All API key sessions closed

🎉 Multi-Key Large SEARCH COMPLETE!
   • Entity cache: output/entity_cache.csv
   • Database: output_multi_key/search_results.db

🔧 MULTI-KEY PROTECTION FINAL STATS

🔑 API Key Pool (Round-Robin)
   • Total API requests made: 2,646
   • Aggregate rate achieved: 851 req/min
   • KEY1: 882 requests (33.3%) — peak 12/12 concurrent
   • KEY2: 882 requests (33.3%) — peak 12/12 concurrent
   • KEY3: 882 requests (33.3%) — peak 12/12 concurrent

🛡️ Per-Key Rate Limiters
   • Limit per key: 475 req/min
   • Throttle events (all keys): 590
   • Total wait time: 295.00s
   • Rate limit warnings (429/403): 0

🔄 Automatic Retry
   • Max retry attempts configured: 5
   • Retry strategy: 1s wait on 429, progressive backoff on 403

✅ All protection layers worked successfully — multi-key pattern ready for production use!


---

## 📖 How to Customize This Notebook

### API Keys

Create a `.env` file with three Bigdata.com API keys (copy from `.env.example`):

```
BIGDATA_API_KEY1=your-api-key-1
BIGDATA_API_KEY2=your-api-key-2
BIGDATA_API_KEY3=your-api-key-3
```

Each key has an independent 500 req/min quota. This notebook routes ~475 req/min per key for a ~1,425 req/min aggregate from a single IP.

### Adding/Modifying Tickers

Edit the `TICKERS_INPUT` variable in the configuration cell.

### Changing Date Range

Edit `START_DATE` and `END_DATE` in the configuration cell:
```python
START_DATE = "2024-12-01"
END_DATE = "2024-12-31"
```

### Adding Custom Topics

Add topics to the `TOPICS` list in the configuration cell:
```python
TOPICS = [
    {"topic_name": "AI Strategy", "topic_text": "AI initiatives {company}?"},
    {"topic_name": "Supply Chain", "topic_text": "supply chain issues {company}?"},
    # Add more topics...
]
```

---

## 📚 API Documentation
- [Bigdata.com API Docs](https://docs.bigdata.com)
- [Knowledge Graph API](https://docs.bigdata.com/getting-started/knowledge_graph)
- [Search API](https://docs.bigdata.com/getting-started/search)

---
